In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

In [ ]:
def sigmoid(x, a1, a2, a3, a4):
    s = 1 + np.exp( -a3*(x-a2) )
    y = a1 / s + a4
    return y

popt = np.array([ 1.76294050e+00, -1.77042245e+01,  3.12374930e-03, -8.83140230e-01])

def eval_to_winodds(x):
    return sigmoid(x, *popt)

mate_to_winodds = {
    -15: -0.844,
    -14: -0.844,
    -13: -0.84,
    -12: -0.852,
    -11: -0.848,
    -10: -0.859,
    -9: -0.85,
    -8: -0.852,
    -7: -0.86,
    -6: -0.869,
    -5: -0.878,
    -4: -0.88,
    -3: -0.89,
    -2: -0.903,
    -1: -0.935,

    1: 0.938,
    2: 0.91,
    3: 0.896,
    4: 0.888,
    5: 0.882,
    6: 0.877,
    7: 0.87,
    8: 0.862,
    9: 0.863,
    10: 0.859,
    11: 0.847,
    12: 0.845,
    13: 0.848,
    14: 0.836,
    15: 0.833
}

## Добавляем признаки

In [ ]:
def generate_features(df_games, df_moves):
    """Создает признаки"""
    
    # !Ошибки
    df_moves["IsInaccuracy"] = df_moves["Move"].str.endswith("?!")
    df_moves["IsBlunder"] = df_moves["Move"].str.endswith("??")
    df_moves["IsMistake"] = df_moves["Move"].str.endswith("?") & (~df_moves["Move"].str.endswith("??"))
    df_moves["IsWrongMove"] = df_moves["IsInaccuracy"] | df_moves["IsBlunder"] | df_moves["IsMistake"]
    df_moves["IsBadMove"] = df_moves["IsBlunder"] | df_moves["IsMistake"]
    df_moves["IsOkayMove"] = ~(df_moves["IsInaccuracy"] | df_moves["IsBlunder"] | df_moves["IsMistake"])
    
    # !Средний ход ошибок
    df_moves["MoveNumberInaccuracy"] = df_moves["MoveNumber"].where(df_moves["IsInaccuracy"])
    df_moves["MoveNumberBlunder"] = df_moves["MoveNumber"].where(df_moves["IsBlunder"])
    df_moves["MoveNumberMistake"] = df_moves["MoveNumber"].where(df_moves["IsMistake"])
    df_moves["MoveNumberWrongMove"] = df_moves["MoveNumber"].where(df_moves["IsWrongMove"])
    df_moves["MoveNumberBadMove"] = df_moves["MoveNumber"].where(df_moves["IsBadMove"])
    df_moves["MoveNumberOkayMove"] = df_moves["MoveNumber"].where(df_moves["IsOkayMove"])
    
    # !Маты
    df_moves["MateIn"] = df_moves["Eval"].where(
        df_moves["Eval"].str.startswith("#"),
        "#0"
    )
    df_moves["MateIn"] = df_moves["MateIn"].str[1:].astype(float)
    # Возьмем маты через 5 ходов
    df_moves["HasMate"] = (df_moves["MateIn"].abs() >= 1) & (df_moves["MateIn"].abs() <= 5)
    
    # !EVAL
    df_moves["EvalCentipawn"] = pd.to_numeric(df_moves["Eval"], errors="coerce").multiply(100).round()
    # Обрежем оценку позиции на +- 12 пешек
    df_moves["EvalCentipawn"] = df_moves["EvalCentipawn"].clip(-1200, 1200)
    # Мат считаем как максимальное преимущество
    df_moves.loc[df_moves["MateIn"] > 0, "EvalCentipawn"] = 1200
    df_moves.loc[df_moves["MateIn"] < 0, "EvalCentipawn"] = -1200
    
    df_moves["AbsEval"] = df_moves["EvalCentipawn"].abs()
    df_moves["IsEqualGame300"] = df_moves["EvalCentipawn"].abs() <= 300
    df_moves["IsLostGame600"] = df_moves["EvalCentipawn"].abs() >= 600
    
    # !Diff
    df_moves["CentipawnLoss"] = df_moves["EvalCentipawn"].diff().abs().fillna(0)
    df_moves.loc[(df_moves["MoveNumber"] == 1), "CentipawnLoss"] = 0
    df_moves["StartCentipawnLoss15"] = df_moves["CentipawnLoss"].where(df_moves["MoveNumber"] <= 15)
    
    # !Шахи
    df_moves["IsCheck"] = df_moves["Move"].str.contains("+", regex=False)
    df_moves["IsCapture"] = df_moves["Move"].str.contains("x", regex=False)
    
    # !В разрезе фигур и ходов
    df_moves["Piece"] = df_moves["Move"].str[0]
    df_moves["KnightCentipawnLoss"] = df_moves["CentipawnLoss"].where(df_moves["Piece"] == "N")
    df_moves["PawnCentipawnLoss"] = df_moves["CentipawnLoss"].where(df_moves["Piece"].str.islower())
    
    
    # WinOdds
    df_moves["WinOdds"] = (
        df_moves["EvalCentipawn"]
        .map(eval_to_winodds)
        .combine_first(
            df_moves["MateIn"]
            .map(mate_to_winodds)
        )
    ).ffill()
    
    df_moves["AdvLost"] = df_moves["WinOdds"].diff().abs()
    df_moves["StartAdvLost10"] = df_moves["AdvLost"].where(df_moves["MoveNumber"] <= 10)
    
    # Time
    df_moves["ClockSeconds"] = (
        df_moves["Clock"]
        .str.split(":", expand=True).astype(int)
        .assign(ClockSeconds=lambda x: x[1]*60 + x[2])["ClockSeconds"]
    )
    df_moves["MoveTime"] = -df_moves["ClockSeconds"].diff(periods=2)
    df_moves.loc[ df_moves["MoveNumber"] == 1, "MoveTime" ] = 0
    
    
    # !Агрегируем
    agg = df_moves.groupby("GameId").agg(
        
        NBlunders=("IsBlunder", "sum"),
        NOkayMoves=("IsOkayMove", "sum"),
        
        MeanBlunders=("IsBlunder", "mean"),
        MeanMistakes=("IsMistake", "mean"),
        MeanBadMoves=("IsBadMove", "mean"),
        MeanOkayMoves=("IsOkayMove", "mean"),
        
        MoveNumberBlunder=("MoveNumberBlunder", "mean"),
        MoveNumberMistake=("MoveNumberMistake", "mean"),
        MoveNumberBadMove=("MoveNumberBadMove", "mean"),
        
        MeanAbsEval=("AbsEval", "mean"),
        EvalStd=("EvalCentipawn", "std"),
        NEqualGame300=("IsEqualGame300", "sum"),
        MeanLostGame600=("IsLostGame600", "mean"),
        
        MeanHasMate=("HasMate", "mean"),
        
        MeanCentipawnLoss=("CentipawnLoss", "mean"),
        StartCentipawnLoss15=("StartCentipawnLoss15", "mean"),        
        
        MeanChecks=("IsCheck", "mean"),
        
        KnightCentipawnLoss=("KnightCentipawnLoss", "mean"),
        PawnCentipawnLoss=("PawnCentipawnLoss", "mean"),
        
        NMoves=("MoveNumber", "max"),
        
        WinOddsStd=("WinOdds", "std"),
        WinOddsMean=("WinOdds", "mean"),
        WinOddsMedian=("WinOdds", "median"),
        MaxAdvLost=("AdvLost", "max"),
        MeanAdvLost=("AdvLost", "mean"),
        MedianAdvLost=("AdvLost", "median"),
        StartAdvLost10=("StartAdvLost10", "mean"),
        
        # new
        TimeSpentMean=("MoveTime", "mean"),
        AbsEvalMedian=("AbsEval", "median"),
        CentipawnLossMedian=("CentipawnLoss", "median")
    )
    
    # NaN filling    
    agg = agg.fillna(0)
    
    get_mean_elo = lambda df: (df["WhiteElo"] + df["BlackElo"]) // 2
    
    df_features = (
        df_games
        .assign(Elo=get_mean_elo)
        .loc[:, ["GameId", "Elo", "Opening", "ECO"]]
    )

    df_features = df_features.merge(agg, on="GameId")
    
    return df_features

In [ ]:
n_batches = 25
for batch in range(1, n_batches + 1):
    
    df_games = pd.read_parquet(f"01_parsed/batch_{batch}_games.parquet")
    df_moves = pd.read_parquet(f"01_parsed/batch_{batch}_moves.parquet")
        
    df_games = df_games[
        (df_games["WhiteRatingDiff"].astype(float).abs() <= 20) &
        (df_games["BlackRatingDiff"].astype(float).abs() <= 20) &
        ((df_games["WhiteElo"] - df_games["BlackElo"]).abs() <= 100)
    ]
    
    df_games = df_games[ df_games["Termination"] == "Normal" ]
    
    ids = set(df_games["GameId"]) & set(df_moves["GameId"])
    df_games = df_games[ df_games["GameId"].isin(ids) ]
    df_moves = df_moves[ df_moves["GameId"].isin(ids) ]
    
    # weird na values
    minor_last_move_error = df_moves[(
        df_moves["Move"].str.contains("#") &
        df_moves["Eval"].isna() &
        df_moves["Clock"].isna()
    )].index
    df_moves = df_moves.drop(index=minor_last_move_error)
    
    df_features = generate_features(df_games, df_moves)

    df_features.to_parquet(f"02_features/batch_{batch}.parquet")
    
    print(f"Batch {batch}: Done")

## Корзина

In [ ]:
# Отфильтрованные признаки

In [ ]:
# def get_quntile(a):
#     return lambda x: np.quantile(x, a)

In [ ]:
# df_moves["IsInaccuracy"] = df_moves["Move"].str.endswith("?!")
# df_moves["IsBlunder"] = df_moves["Move"].str.endswith("??")
# df_moves["IsMistake"] = df_moves["Move"].str.endswith("?") & (~df_moves["Move"].str.endswith("??"))
# df_moves["IsWrongMove"] = df_moves["IsInaccuracy"] | df_moves["IsBlunder"] | df_moves["IsMistake"]
# df_moves["IsBadMove"] = df_moves["IsBlunder"] | df_moves["IsMistake"]
# df_moves["IsOkayMove"] = ~(df_moves["IsInaccuracy"] | df_moves["IsBlunder"] | df_moves["IsMistake"])

# NInaccuracies=("IsInaccuracy", "sum"),
# NBlunbers=("IsBlunder", "sum"),
# NMistakes=("IsMistake", "sum"),
# NWrongMoves=("IsWrongMove", "sum"),
# NBadMoves=("IsBadMove", "sum"),
# NOkayMoves=("IsOkayMove", "sum"),

# MeanInaccuracies=("IsInaccuracy", "mean"),
# MeanBlunbers=("IsBlunder", "mean"),
# MeanMistakes=("IsMistake", "mean"),
# MeanWrongMoves=("IsWrongMove", "mean"),
# MeanBadMoves=("IsBadMove", "mean"),
# MeanOkayMoves=("IsOkayMove", "mean")

In [ ]:
# df_moves["MoveNumberInaccuracy"] = where(df_moves["IsInaccuracy"], df_moves["MoveNumber"])
# df_moves["MoveNumberBlunder"] = where(df_moves["IsBlunder"], df_moves["MoveNumber"])
# df_moves["MoveNumberMistake"] = where(df_moves["IsMistake"], df_moves["MoveNumber"])
# df_moves["MoveNumberWrongMove"] = where(df_moves["IsWrongMove"], df_moves["MoveNumber"])
# df_moves["MoveNumberBadMove"] = where(df_moves["IsBadMove"], df_moves["MoveNumber"])
# df_moves["MoveNumberOkayMove"] = where(df_moves["IsOkayMove"], df_moves["MoveNumber"])

# MoveNumberInaccuracy=("MoveNumberInaccuracy", "mean"),
# MoveNumberBlunder=("MoveNumberBlunder", "mean"),
# MoveNumberMistake=("MoveNumberMistake", "mean"),
# MoveNumberWrongMove=("MoveNumberWrongMove", "mean"),
# MoveNumberBadMove=("MoveNumberBadMove", "mean"),
# MoveNumberOkayMove=("MoveNumberOkayMove", "mean"),

In [ ]:
# df_moves["AbsEval"] = df_moves["EvalCentipawn"].abs()
# df_moves["IsEqualGame100"] = df_moves["EvalCentipawn"].abs() <= 100
# df_moves["IsEqualGame200"] = df_moves["EvalCentipawn"].abs() <= 200
# df_moves["IsEqualGame300"] = df_moves["EvalCentipawn"].abs() <= 300

# MeanAbsEval=("AbsEval", "mean"),
# NEqualGame100=("IsEqualGame100", "sum"),
# NEqualGame200=("IsEqualGame200", "sum"),
# NEqualGame300=("IsEqualGame300", "sum"),
# MeanEqualGame100=("IsEqualGame100", "mean"),
# MeanEqualGame200=("IsEqualGame200", "mean"),
# MeanEqualGame300=("IsEqualGame300", "mean"),

In [ ]:
# df_moves["IsLostGame800"] = df_moves["EvalCentipawn"].abs() >= 800
# df_moves["IsLostGame700"] = df_moves["EvalCentipawn"].abs() >= 700
# df_moves["IsLostGame600"] = df_moves["EvalCentipawn"].abs() >= 600

# NLostGame800=("IsLostGame800", "sum"),
# NLostGame700=("IsLostGame700", "sum"),
# NLostGame600=("IsLostGame600", "sum"),
# MeanLostGame800=("IsLostGame800", "mean"),
# MeanLostGame700=("IsLostGame700", "mean"),
# MeanLostGame600=("IsLostGame600", "mean"),

In [ ]:
# df_moves["IsCheck"] = df_moves["Move"].str.contains("+", regex=False)
# df_moves["IsCapture"] = df_moves["Move"].str.contains("x", regex=False)
# NChecks=("IsCheck", "sum")
# NCaptures=("IsCapture", "sum"),
# MeanCaptures=("IsCapture", "mean")

In [ ]:
# df_moves["Piece"] = df_moves["Move"].str[0]
# df_moves["QueenCentipawnLoss"] = where(df_moves["Piece"] == "Q", df_moves["CentipawnLoss"])
# df_moves["RookCentipawnLoss"] = where(df_moves["Piece"] == "R", df_moves["CentipawnLoss"])
# df_moves["KnightCentipawnLoss"] = where(df_moves["Piece"] == "N", df_moves["CentipawnLoss"])
# df_moves["BishopCentipawnLoss"] = where(df_moves["Piece"] == "B", df_moves["CentipawnLoss"])
# df_moves["CheckCentipawnLoss"] = where(df_moves["IsCheck"], df_moves["CentipawnLoss"])
# df_moves["CaptureCentipawnLoss"] = where(df_moves["IsCheck"], df_moves["CentipawnLoss"])

# QueenCentipawnLoss=("QueenCentipawnLoss", "mean"),
# RookCentipawnLoss=("RookCentipawnLoss", "mean"),
# KnightCentipawnLoss=("KnightCentipawnLoss", "mean"),
# BishopCentipawnLoss=("BishopCentipawnLoss", "mean"),
# CheckCentipawnLoss=("CheckCentipawnLoss", "mean"),
# CaptureCentipawnLoss=("CaptureCentipawnLoss", "mean"),

In [ ]:
# df_moves["EarlyCentipawnLoss10"] = where(df_moves["MoveNumber"] <= 10, df_moves["CentipawnLoss"])
# df_moves["EarlyCentipawnLoss20"] = where(df_moves["MoveNumber"] <= 20, df_moves["CentipawnLoss"])
# df_moves["EarlyCentipawnLoss30"] = where(df_moves["MoveNumber"] <= 30, df_moves["CentipawnLoss"])
# df_moves["EarlyCentipawnLoss40"] = where(df_moves["MoveNumber"] <= 40, df_moves["CentipawnLoss"])
# df_moves["EarlyCentipawnLoss50"] = where(df_moves["MoveNumber"] <= 50, df_moves["CentipawnLoss"])
# df_moves["EarlyCentipawnLoss60"] = where(df_moves["MoveNumber"] <= 60, df_moves["CentipawnLoss"])

# EarlyCentipawnLoss30=("EarlyCentipawnLoss30", "mean"),
# EarlyCentipawnLoss40=("EarlyCentipawnLoss40", "mean"),
# EarlyCentipawnLoss50=("EarlyCentipawnLoss50", "mean"),
# EarlyCentipawnLoss60=("EarlyCentipawnLoss60", "mean"),

In [ ]:
# df_moves["EarlyCentipawnLoss5"] = where(df_moves["MoveNumber"] <= 5, df_moves["CentipawnLoss"])
# df_moves["EarlyCentipawnLoss10"] = where(df_moves["MoveNumber"] <= 10, df_moves["CentipawnLoss"])
# df_moves["EarlyCentipawnLoss15"] = where(df_moves["MoveNumber"] <= 15, df_moves["CentipawnLoss"])
# df_moves["MiddleCentipawnLoss1020"] = where((df_moves["MoveNumber"] > 10) & (df_moves["MoveNumber"] <= 20), df_moves["CentipawnLoss"])

# EarlyCentipawnLoss10=("EarlyCentipawnLoss10", "mean"),
# MiddleCentipawnLoss1020=("MiddleCentipawnLoss1020", "mean"),
# EarlyCentipawnLoss5=("EarlyCentipawnLoss5", "mean"),
# EarlyCentipawnLoss15=("EarlyCentipawnLoss15", "mean"),   

## Winodds

In [ ]:
df_games = pd.concat((
    pd.read_parquet(f"parsed/batch_{i}_games.parquet")
    for i in range(1, 2 + 1)
))

df_moves = pd.concat((
    pd.read_parquet(f"parsed/batch_{i}_moves.parquet")
    for i in range(1, 2 + 1)
))

In [ ]:
 df_moves["MateIn"] = df_moves["Eval"].where(
    df_moves["Eval"].str.startswith("#")
)
df_moves["MateIn"] = df_moves["MateIn"].str[1:].astype(float)
df_moves["MateIn"] = df_moves["MateIn"].clip(-15, 15)

In [ ]:
df_moves["MateIn"].map(mate_to_winodds)

In [ ]:
df_moves["EvalCentipawn"] = pd.to_numeric(df_moves["Eval"], errors="coerce").multiply(100).round()

In [ ]:
df_moves["WinOdds"] = df_moves["EvalCentipawn"].map(eval_to_winodds).combine_first(
    df_moves["MateIn"].map(mate_to_winodds)
).ffill()

In [ ]:
px.line(
    df_moves[
        df_moves["GameId"] == df_moves.sample(1)["GameId"].values[0]
    ]["WinOdds"]
)

In [ ]:
df_games["Result"].value_counts(normalize=True).to_dict()

In [ ]:
df_games["WhiteWon"] = df_games["Result"].map({"1-0": 1, "0-1": -1, "1/2-1/2": 0})

In [ ]:
df_games[["GameId", "WhiteWon"]]

In [ ]:
df_moves = df_moves.merge(
    df_games[["GameId", "WhiteWon"]],
    on="GameId"
)

In [ ]:
df_moves["EvalCentipawn"] = pd.to_numeric(df_moves["Eval"], errors="coerce").multiply(100).round()
df_moves["EvalCentipawn"] = df_moves["EvalCentipawn"].clip(-2000, 2000)

In [ ]:
px.histogram(
    df_moves["EvalCentipawn"] // 50
)

In [ ]:
from scipy.optimize import curve_fit

xdata = a.index
ydata = a.values

def sigmoid(x, a1, a2, a3, a4):
    s = 1 + np.exp( -a3*(x-a2) )
    y = a1 / s + a4
    return y

p0 = [2, -8, 0.003, -0.8]

popt, _ = curve_fit(sigmoid, xdata, ydata, p0, method="lm")

In [ ]:
a = df_moves.groupby(
    df_moves["EvalCentipawn"] // 50 * 50
).agg({"WhiteWon": "mean"}).squeeze()

fig = px.line(
    a,
    template="plotly_white"
)

fig.add_traces(px.line(
    x=np.linspace(-2000, 2000, 100),
    y=eval_to_winodds(np.linspace(-2000, 2000, 100)),
).data[0])

fig.data[0].line.color="lightgrey"

fig.show()

In [ ]:
df_moves["MateIn"] = df_moves["Eval"].where(
    df_moves["Eval"].str.startswith("#"),
    "#0"
)
df_moves["MateIn"] = df_moves["MateIn"].str[1:].astype(int)
df_moves["MateIn"] = df_moves["MateIn"].clip(-15, 15)

In [ ]:
px.histogram(
    df_moves["MateIn"]
)

In [ ]:
b = df_moves.groupby(
        df_moves["MateIn"]
    ).agg({"WhiteWon": "mean"}).squeeze()

px.line(
    b,
    template="plotly_white"
)

In [ ]:
b.round(3).to_dict()

## Time

In [ ]:
df = pd.read_parquet(f"parsed/batch_{2}_moves.parquet")

In [ ]:
df["ClockSeconds"] = (
    df["Clock"]
    .str.split(":", expand=True).astype(int)
    .assign(ClockSeconds=lambda x: x[1]*60 + x[2])["ClockSeconds"]
)

In [ ]:
df["MoveTime"] = -df["ClockSeconds"].diff(periods=2)
df.loc[ df["MoveNumber"] == 1, "MoveTime" ] = 0

In [ ]:
df.

## TS Clustering

In [ ]:
df_games = pd.concat((
    pd.read_parquet(f"parsed/batch_{i}_games.parquet")
    for i in range(1, 5 + 1)
))

df_moves = pd.concat((
    pd.read_parquet(f"parsed/batch_{i}_moves.parquet")
    for i in range(1, 5 + 1)
))

In [ ]:
df_moves["MateIn"] = df_moves["Eval"].where(
    df_moves["Eval"].str.startswith("#"),
    "#0"
)
df_moves["MateIn"] = df_moves["MateIn"].str[1:].astype(float)

df_moves["EvalCentipawn"] = pd.to_numeric(df_moves["Eval"], errors="coerce").multiply(100).round()
df_moves["EvalCentipawn"] = df_moves["EvalCentipawn"].clip(-1200, 1200)
df_moves.loc[df_moves["MateIn"] > 0, "EvalCentipawn"] = 1200
df_moves.loc[df_moves["MateIn"] < 0, "EvalCentipawn"] = -1200

df_moves["WinOdds"] = (
    df_moves["EvalCentipawn"]
    .map(eval_to_winodds)
    .combine_first(
        df_moves["MateIn"]
        .map(mate_to_winodds)
    )
).ffill()

In [ ]:
df_games["GameId"] = df_games["GameId"].astype("category")
df_moves["GameId"] = df_moves["GameId"].astype("category")

In [ ]:
games = []
ratings = []
ids = []
for i in range(100000):
    if i % 100 == 0:
        print(i, end="\r")
    
    r = df_games.sample(1).T.squeeze()
    game = df_moves[
        df_moves["GameId"] == r["GameId"]
    ]
    if len(game) < 3:
        continue
    else:
        game = game.head(40)
        
    games.append(game["Move"].values)
    ratings.append( (r["WhiteElo"]+r["BlackElo"])//2 )
    ids.append( r["GameId"] )

In [ ]:
df_series = pd.DataFrame(games).assign(Elo=ratings).assign(GameId=ids)
df_series = df_series.dropna()

In [ ]:
df_series

In [ ]:
moves = df_series[0]
for i in range(1, 6):
    moves = moves + " " + df_series[i]

df_series["moves"] = moves

In [ ]:
g = (
    df_series
    .groupby("moves").agg({"Elo": ["count", "mean"]})["Elo"]
    .sort_values("mean", ascending=False)
)

g = g[ g["count"] > 10 ]
g = g.sort_values("mean", ascending=False)

In [ ]:
px.line(
    g["mean"].reset_index(drop=True),
    template="plotly_white"
)

In [ ]:
df_series[
    df_series["moves"] == g.index[-4]
].sample(1)